In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [7]:
!pip install dagshub mlflow
import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment2', mlflow=True)

import mlflow
mlflow.set_experiment("DecisionTree_Training")
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c09ab762-28e5-4503-afb7-aec9e4465e60&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=038c2d4911d0c6ee913c3db5633debd5d1cd36ff6937371dc94f6ada8062f1ce




Output()

Accessing as smama23

Initialized MLflow to track repo "smama23/MLassignment2"

Repository smama23/MLassignment2 initialized!

🏃 View run bright-toad-480 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/199d91c7b31d4777897cbc065e85dd7d
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


In [8]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

Cleaning

In [9]:
with mlflow.start_run(run_name="DecisionTree_Cleaning_v1"):
    
    df_clean = df.copy()
    
    y = df_clean["isFraud"]
    df_clean = df_clean.drop(columns=["isFraud"])
    
    df_clean = df_clean.fillna(-999)
    
    mlflow.log_param("cleaning", "fillna_-999")
    mlflow.log_metric("num_features", df_clean.shape[1])

🏃 View run DecisionTree_Cleaning_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/37d16d5b62b54d538a05785d604142f0
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


Feature Engineering

In [10]:
with mlflow.start_run(run_name="DecisionTree_FeatureEngineering_v1"):
    
    df_fe = df_clean.copy()
    
    df_fe["hour"] = (df_fe["TransactionDT"] // 3600) % 24
    df_fe["day"] = (df_fe["TransactionDT"] // (3600 * 24)) % 7
    df_fe["log_amount"] = np.log1p(df_fe["TransactionAmt"])
    
    mlflow.log_param("features_added", "hour, day, log_amount")
    mlflow.log_metric("num_features_after_fe", df_fe.shape[1])

🏃 View run DecisionTree_FeatureEngineering_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/78a6e447dc764ac09a09f5f200d9c157
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


In [11]:
def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)
    return df

Feature Selection

In [12]:
with mlflow.start_run(run_name="DecisonTree_FeatureSelection_v1"):
    
    df_enc = df_fe.copy()
    
    cat_cols = df_enc.select_dtypes(include="object").columns
    df_enc = frequency_encoding(df_enc, cat_cols)
    
    nunique = df_enc.nunique()
    df_fs = df_enc[nunique[nunique > 1].index]
    
    mlflow.log_param("encoding", "frequency")
    mlflow.log_param("feature_selection", "remove_constant")
    mlflow.log_metric("num_features_final", df_fs.shape[1])

🏃 View run DecisonTree_FeatureSelection_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/f95ccb55f92d41a9a809e9bcaef32f59
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Training

In [15]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="DecisionTree_Training_v1"):

    X_train, X_val, y_train, y_val = train_test_split(
        df_fs, y, test_size=0.2, stratify=y, random_state=42
    )
    
    model = DecisionTreeClassifier(
        max_depth=None,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("max_depth", "None")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/03 20:41:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 20:41:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


1.0 0.7932787810350802
🏃 View run DecisionTree_Training_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/4907c565ba234799a93944c89d06de51
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


In [16]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="DecisionTree_Training_v2_tuned"):
    
    model = DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 20)
    mlflow.log_param("min_samples_leaf", 10)
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/03 20:42:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 20:42:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8203776297265869 0.8134467813378026
🏃 View run DecisionTree_Training_v2_tuned at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/46452d23745c4e8189c69405c8707b9e
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


In [17]:
feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
top_features = feature_importance.sort_values(ascending=False).head(20)

top_features

V258             0.321756
C1               0.106392
C14              0.059345
C12              0.057550
C13              0.052450
V149             0.043468
V317             0.027364
id_17            0.017822
card2            0.016040
V87              0.012961
addr1            0.011001
V283             0.010309
V62              0.009998
V152             0.009693
TransactionDT    0.009102
V130             0.008960
C2               0.008781
id_01            0.008266
V323             0.008238
V45              0.007975
dtype: float64

In [18]:
mlflow.log_dict(top_features.to_dict(), "dt_top_features.json")

In [19]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            freq = X[col].value_counts() / len(X)
            self.freq_maps[col] = freq
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["hour"] = X["TransactionDT"]
        X["day"] = X["TransactionDT"]
        X["log_amount"] = np.log1p(X["TransactionAmt"])
        return X


class FillNaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=-999):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.fillna(self.fill_value)

In [20]:
cat_cols = df.drop(columns=["isFraud"]).select_dtypes(include="object").columns

In [21]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("cleaning", FillNaTransformer()),
    ("feature_engineering", FeatureEngineer()),
    ("encoding", FrequencyEncoder(cat_cols)),
    ("model", DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42
    ))
])

In [22]:
mlflow.end_run(status="FINISHED")

with mlflow.start_run(run_name="DecisionTree_Pipeline_v1"):
    
    X = df.drop(columns=["isFraud"])
    y = df["isFraud"]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    pipeline.fit(X_train, y_train)
    
    y_train_pred = pipeline.predict_proba(X_train)[:, 1]
    y_val_pred = pipeline.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("model", "DecisionTree_pipeline")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(pipeline, "rf_pipeline_model")
    
    print(train_auc, val_auc)

🏃 View run unruly-perch-691 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/317c1af7b6274a46964634d6e24aa8c4
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1


2026/05/03 20:46:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 20:46:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8203943950521009 0.8125969859274962
🏃 View run DecisionTree_Pipeline_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1/runs/8d48b527fa824142b2bd37c4f7e4db46
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/1
